In [2]:
# # POS Tagging in NLP

# POS Tagging means Part of Speech Tagging.
# It is an NLP technique where each word in a sentence is assigned its grammatical role such as:
# Noun
# Verb
# Adjective
# Adverb
# Pronoun
# Preposition
# Conjunction


In [3]:
# Why POS Tagging is Important?

# POS Tagging is used in:
# Text Classification
# Chatbots
# Machine Translation
# Named Entity Recognition (NER)
# Sentiment Analysis
# Speech Recognition
# Grammar Checking

In [4]:
# Common POS Tags

# | POS Tag     | Meaning
# |------------ |------------
# | NN          | Noun
# | NNS         | Plural noun
# | NNP         | proper noun
# | VB          | verb
# | VBD         | verb past tense 
# | VBG         | verb gerund
# | JJ          | adjective 
# | RB          | adverb 
# | PRP         | Pronoun
# |IN           | Preposition



In [11]:
! pip install spacy

In [15]:
! python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     -- ------------------------------------- 0.8/12.8 MB 8.5 MB/s eta 0:00:02
     --------- ------------------------------ 3.1/12.8 MB 11.5 MB/s eta 0:00:01
     ----------------- ---------------------- 5.5/12.8 MB 11.2 MB/s eta 0:00:01
     ------------------------ --------------- 7.9/12.8 MB 11.6 MB/s eta 0:00:01
     ------------------------------- ------- 10.2/12.8 MB 11.4 MB/s eta 0:00:01
     ----------------------------------- --- 11.8/12.8 MB 11.0 MB/s eta 0:00:01
     ---------------------------------------- 12.8/12.8 MB 10.7 MB/s  0:00:01
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


In [16]:
!pip install click

In [17]:
import spacy

# Load English model
nlp = spacy.load("en_core_web_sm")

# Input text
doc = nlp("Saurabh is learning NLP")

# POS tagging
for token in doc:
    print(token.text, " ---- >", token.pos_)

Saurabh  ---- > PROPN
is  ---- > AUX
learning  ---- > VERB
NLP  ---- > PROPN


In [ ]:
# ExampLe -2

import spacy
# Load model
nlp = spacy.load("en_core_web_sm")

text = "Saurabh plays cricket"
doc = nlp(text)
for token in doc:
    print(token.text, " ---- >", token.pos_)

ModuleNotFoundError: No module named 'spacy'

In [9]:
# Example -3

import spacy

nlp = spacy.load("en_core_web_sm")

text = "Saurabh is learning NLP"

doc = nlp(text)

for token in doc:
    print(
    "Word:", token.text,
    "| POS:", token. pos_,
    "| TAG:", token.tag_
    )

ModuleNotFoundError: No module named 'click'

In [10]:
import spacy
from spacy import displacy

nlp = spacy.load("en_core_web_sm")

text = "Saurabh is learning NLP"

doc = nlp(text)

displacy.serve(doc, style="dep")

ModuleNotFoundError: No module named 'click'

In [26]:
import spacy
import json
from textblob import TextBlob

nlp = spacy.load("en_core_web_sm")

class AspectSentimentAnalyzer:
    """
    Real-world ABSA using:
    - spaCy POS tagging (NOUN = feature, ADJ/VERB = opinion)
    - Dependency parsing to link opinions to their aspects
    - TextBlob polarity for sentiment scoring (clause-level, negation-aware)
    - Negation flip only applied at word-level fallback
    """

    def __init__(self):
        self.FEATURE_POS = {"NOUN", "PROPN"}
        self.OPINION_POS = {"ADJ", "ADV", "VERB"}
        self.NEGATIONS   = {"not", "no", "never", "n't", "hardly", "barely"}
        self.SKIP_NOUNS  = {"phone", "product", "device", "thing", "it", "one"}

    # ------------------------------------------------------------------ #
    #  CORE: link opinion words to their head noun via dependency tree    #
    # ------------------------------------------------------------------ #
    def _extract_pairs(self, doc):
        """
        Walk the dependency tree.
        Patterns caught:
          amod  → "amazing camera"        (ADJ modifies NOUN)
          nsubj → "battery drains fast"   (NOUN is subject of VERB)
          acomp → "camera is stunning"    (ADJ complement of copula)
          advmod→ "drains quickly"        (ADV modifies VERB)
        """
        pairs = []

        for token in doc:
            aspect  = None
            opinion = None

            # Pattern 1: adjective directly modifying a noun (amod)
            if token.dep_ == "amod" and token.head.pos_ in self.FEATURE_POS:
                aspect  = token.head
                opinion = token

            # Pattern 2: noun is subject of a verb (nsubj)
            elif token.dep_ == "nsubj" and token.head.pos_ == "VERB":
                aspect  = token
                opinion = token.head

            # Pattern 3: adjective complement of a copula (acomp / attr)
            elif token.dep_ in {"acomp", "attr"} and token.pos_ in self.OPINION_POS:
                opinion = token
                for child in token.head.children:
                    if child.dep_ == "nsubj" and child.pos_ in self.FEATURE_POS:
                        aspect = child
                        break

            if aspect and opinion and aspect.lemma_.lower() not in self.SKIP_NOUNS:
                pairs.append((aspect, opinion))

        return pairs

    # ------------------------------------------------------------------ #
    #  NEGATION: check if opinion token has a negation in its subtree     #
    # ------------------------------------------------------------------ #
    def _is_negated(self, opinion_token):
        for child in opinion_token.head.children:
            if child.dep_ == "neg" or child.text.lower() in self.NEGATIONS:
                return True
        return False

    # ------------------------------------------------------------------ #
    #  SENTIMENT: clause-level TextBlob first, word-level fallback        #
    # ------------------------------------------------------------------ #
    def _get_sentiment(self, opinion_token, clause_text):
        clause_polarity = TextBlob(clause_text).sentiment.polarity
        word_polarity   = TextBlob(opinion_token.lemma_).sentiment.polarity
        negated         = self._is_negated(opinion_token)

        if clause_polarity != 0.0:
            # TextBlob already handles negation at clause level — trust it directly
            # avoids double-negation bug (e.g. "not bad" → TextBlob gives +0.35,
            # manual flip would wrongly make it -0.35)
            polarity = clause_polarity
        elif word_polarity != 0.0:
            # clause is neutral, fall back to word-level + manual negation flip
            polarity = -word_polarity if negated else word_polarity
        else:
            polarity = 0.0

        if polarity > 0.05:
            label = "positive"
        elif polarity < -0.05:
            label = "negative"
        else:
            label = "neutral"

        return label, round(polarity, 3)

    # ------------------------------------------------------------------ #
    #  PUBLIC API                                                          #
    # ------------------------------------------------------------------ #
    def analyze(self, review: str) -> list[dict]:
        doc = nlp(review)
        results = []
        seen = set()

        for sent in doc.sents:
            pairs = self._extract_pairs(sent.as_doc())
            for aspect, opinion in pairs:
                key = (aspect.lemma_.lower(), opinion.lemma_.lower())
                if key in seen:
                    continue
                seen.add(key)

                sentiment_label, polarity = self._get_sentiment(opinion, sent.text)

                results.append({
                    "feature":   aspect.lemma_.lower(),
                    "opinion":   opinion.text.lower(),
                    "pos_tag":   f"{aspect.pos_} → {opinion.pos_}",
                    "dep_rel":   opinion.dep_,
                    "sentiment": sentiment_label,
                    "polarity":  polarity
                })

        return results

    def analyze_to_json(self, review: str, indent: int = 2) -> str:
        return json.dumps({
            "review":  review,
            "aspects": self.analyze(review)
        }, indent=indent)


# ------------------------------------------------------------------ #
#  Run it                                                              #
# ------------------------------------------------------------------ #
analyzer = AspectSentimentAnalyzer()

review = "The camera quality is amazing but the battery drains quickly. Display is not bad at all."
print(analyzer.analyze_to_json(review))

{
  "review": "The camera quality is amazing but the battery drains quickly. Display is not bad at all.",
  "aspects": [
    {
      "feature": "quality",
      "opinion": "amazing",
      "pos_tag": "NOUN \u2192 ADJ",
      "dep_rel": "acomp",
      "sentiment": "positive",
      "polarity": 0.467
    },
    {
      "feature": "battery",
      "opinion": "drains",
      "pos_tag": "NOUN \u2192 VERB",
      "dep_rel": "conj",
      "sentiment": "positive",
      "polarity": 0.467
    },
    {
      "feature": "display",
      "opinion": "bad",
      "pos_tag": "PROPN \u2192 ADJ",
      "dep_rel": "acomp",
      "sentiment": "positive",
      "polarity": 0.35
    }
  ]
}


In [24]:
"""
KeywordExtractor — NLP-based keyword extraction from documents.

Approach:
  - Tokenization + lowercasing
  - Stop word removal (built-in list, no external dependency)
  - TF-IDF-style scoring (term freq × inverse-doc-freq approximation)
  - Positional bonus (keywords near sentence start score higher)
  - Length bonus (longer words tend to be more content-bearing)
  - Optional: bigram extraction, spaCy NER integration

Usage:
  extractor = KeywordExtractor(top_n=10)
  keywords = extractor.extract("Machine learning is transforming healthcare industry")
  print(keywords)  # ['machine learning', 'healthcare', 'industry', ...]

  # With spaCy (richer POS tagging + NER):
  extractor = KeywordExtractor(use_spacy=True)
  result = extractor.extract_detailed("...")
"""

import re
import math
from collections import Counter
from typing import Optional


STOP_WORDS = {
    "a", "an", "the", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "from", "is", "are", "was", "were", "be", "been",
    "being", "have", "has", "had", "do", "does", "did", "will", "would",
    "could", "should", "may", "might", "shall", "can", "this", "that",
    "these", "those", "it", "its", "they", "their", "them", "we", "our",
    "you", "your", "he", "his", "she", "her", "i", "my", "me", "us",
    "who", "which", "what", "when", "where", "how", "why", "not", "no",
    "so", "as", "if", "then", "than", "more", "most", "very", "just",
    "also", "about", "into", "over", "after", "before", "through",
    "between", "during", "some", "all", "any", "each", "both", "many",
    "much", "such", "up", "out", "while", "however", "therefore", "thus",
    "there", "here", "now", "still", "yet", "already", "even", "well",
    "back", "way", "used", "make", "made", "take", "give", "go", "come",
    "said", "see", "get", "got", "use", "using", "new", "old", "one",
    "two", "three", "first", "last", "next", "own", "same", "other",
}

VERB_SUFFIXES = ("ing", "tion", "ize", "ise", "ify", "ate")
NOUN_SUFFIXES = ("ment", "ness", "ity", "ism", "ist", "ance", "ence",
                 "ship", "hood", "er", "or", "ology", "graph")


class KeywordExtractor:
    """
    Extract important keywords from a document using NLP techniques.

    Parameters
    ----------
    top_n : int
        Number of top keywords to return (default 10).
    include_bigrams : bool
        Whether to extract two-word phrases (default True).
    include_verbs : bool
        Whether to include action verbs (default False).
    min_word_len : int
        Minimum character length of a keyword (default 3).
    use_spacy : bool
        If True, use spaCy for POS tagging and NER (requires: pip install spacy
        and: python -m spacy download en_core_web_sm). Defaults to False.
    """

    def __init__(
        self,
        top_n: int = 10,
        include_bigrams: bool = True,
        include_verbs: bool = False,
        min_word_len: int = 3,
        use_spacy: bool = False,
    ):
        self.top_n = top_n
        self.include_bigrams = include_bigrams
        self.include_verbs = include_verbs
        self.min_word_len = min_word_len
        self.use_spacy = use_spacy
        self._nlp = None

        if use_spacy:
            self._load_spacy()

    # ------------------------------------------------------------------ #
    # Public API
    # ------------------------------------------------------------------ #

    def extract(self, text: str) -> list[str]:
        """
        Extract keywords from text. Returns a flat list of keyword strings.

        Example
        -------
        >>> extractor = KeywordExtractor(top_n=5)
        >>> extractor.extract("Machine learning is transforming healthcare industry")
        ['machine learning', 'healthcare', 'industry', 'transforming', ...]
        """
        results = self.extract_detailed(text)
        return [item["keyword"] for item in results]

    def extract_detailed(self, text: str) -> list[dict]:
        """
        Extract keywords with scores, frequencies, and types.

        Returns a list of dicts:
          [{"keyword": str, "score": float, "freq": int, "type": str}, ...]
        """
        if self.use_spacy and self._nlp:
            return self._extract_spacy(text)
        return self._extract_rule_based(text)

    def extract_from_file(self, filepath: str) -> list[str]:
        """Load a .txt file and extract keywords from its contents."""
        with open(filepath, "r", encoding="utf-8") as f:
            return self.extract(f.read())

    # ------------------------------------------------------------------ #
    # Rule-based extraction (no external deps)
    # ------------------------------------------------------------------ #

    def _extract_rule_based(self, text: str) -> list[dict]:
        tokens = self._tokenize(text)
        if not tokens:
            return []

        total = len(tokens)
        freq = Counter(tokens)
        positions: dict[str, list[int]] = {}
        for i, t in enumerate(tokens):
            positions.setdefault(t, []).append(i)

        candidates = []

        # Bigrams first (so constituent unigrams can be deprioritised)
        bigram_parts: set[str] = set()
        if self.include_bigrams:
            bigrams = self._get_bigrams(tokens)
            for phrase, f in bigrams.items():
                score = self._score(phrase, f, total, positions.get(phrase.split()[0], []), is_bigram=True)
                candidates.append({"keyword": phrase, "score": score, "freq": f, "type": "bigram"})
                bigram_parts.update(phrase.split())

        # Unigrams
        for word, f in freq.items():
            if word in bigram_parts and self.include_bigrams:
                continue  # already captured in a bigram
            if not self._is_content_word(word, f):
                continue
            word_type = self._classify(word)
            if word_type == "verb" and not self.include_verbs:
                continue
            score = self._score(word, f, total, positions.get(word, []))
            candidates.append({"keyword": word, "score": score, "freq": f, "type": word_type})

        # Sort and return top N
        candidates.sort(key=lambda x: x["score"], reverse=True)
        return candidates[: self.top_n]

    def _tokenize(self, text: str) -> list[str]:
        text = text.lower()
        text = re.sub(r"[^a-z0-9\s\-]", " ", text)
        return [t for t in text.split() if len(t) >= self.min_word_len]

    def _is_content_word(self, word: str, freq: int) -> bool:
        if word in STOP_WORDS:
            return False
        if len(word) < self.min_word_len:
            return False
        return True

    def _classify(self, word: str) -> str:
        if any(word.endswith(s) for s in VERB_SUFFIXES):
            return "verb"
        if any(word.endswith(s) for s in NOUN_SUFFIXES):
            return "noun"
        if word.endswith(("al", "ive", "ous", "ful", "less", "able", "ible", "ical")):
            return "adj"
        return "noun"  # default assumption

    # Words that look like nouns/adjectives but are actually discourse connectors
    _WEAK_WORDS = {
        "like", "also", "just", "even", "very", "every", "each", "both",
        "many", "much", "such", "less", "more", "most", "only", "same",
        "another", "other", "others", "whether", "although", "since", "because",
    }

    def _get_bigrams(self, tokens: list[str]) -> Counter:
        bigrams: Counter = Counter()
        for i in range(len(tokens) - 1):
            a, b = tokens[i], tokens[i + 1]
            if (
                a not in STOP_WORDS
                and b not in STOP_WORDS
                and a not in self._weak_words()
                and b not in self._weak_words()
                and len(a) >= max(self.min_word_len, 4)
                and len(b) >= max(self.min_word_len, 4)
                # At least one word should look like a noun or technical term
                and (len(a) > 5 or len(b) > 5)
            ):
                bigrams[f"{a} {b}"] += 1
        return bigrams

    def _weak_words(self) -> set:
        return self._WEAK_WORDS

    def _score(
        self,
        word: str,
        freq: int,
        total: int,
        positions: list[int],
        is_bigram: bool = False,
    ) -> float:
        tf = freq / max(total, 1)
        # Approximate IDF — words with higher freq in a single doc get mild penalty
        idf = math.log((total + 1) / (freq + 1)) + 1
        base = tf * idf

        # Positional bonus: early-sentence words are often more important
        pos_bonus = 0.3 if positions and positions[0] < total * 0.2 else 0.0

        # Length bonus: longer words tend to be more specific / meaningful
        len_bonus = 0.15 if len(word) > 8 else (0.08 if len(word) > 5 else 0.0)

        # Bigram bonus: phrases carry more meaning than individual tokens
        bigram_bonus = 0.4 if is_bigram else 0.0

        # Frequency bonus: repeated words are probably important
        freq_bonus = min(freq * 0.1, 0.5)

        return base + pos_bonus + len_bonus + bigram_bonus + freq_bonus

    # ------------------------------------------------------------------ #
    # spaCy-based extraction (richer, requires pip install spacy)
    # ------------------------------------------------------------------ #

    def _load_spacy(self):
        try:
            import spacy
            try:
                self._nlp = spacy.load("en_core_web_sm")
            except OSError:
                print(
                    "[KeywordExtractor] spaCy model not found. Run:\n"
                    "  python -m spacy download en_core_web_sm\n"
                    "Falling back to rule-based extraction."
                )
                self._nlp = None
        except ImportError:
            print(
                "[KeywordExtractor] spaCy not installed. Run:\n"
                "  pip install spacy\n"
                "Falling back to rule-based extraction."
            )
            self._nlp = None

    def _extract_spacy(self, text: str) -> list[dict]:
        doc = self._nlp(text)
        total = len(doc)
        candidates: dict[str, dict] = {}

        allowed_pos = {"NOUN", "PROPN", "ADJ"}
        if self.include_verbs:
            allowed_pos.add("VERB")

        for i, token in enumerate(doc):
            word = token.lemma_.lower()
            if (
                token.pos_ not in allowed_pos
                or token.is_stop
                or token.is_punct
                or len(word) < self.min_word_len
                or word in STOP_WORDS
            ):
                continue

            if word not in candidates:
                candidates[word] = {"keyword": word, "freq": 0, "pos": i, "type": token.pos_.lower()}
            candidates[word]["freq"] += 1

        # Named entities as bigrams
        if self.include_bigrams:
            for ent in doc.ents:
                phrase = ent.text.lower().strip()
                if len(phrase.split()) > 1 and len(phrase) >= self.min_word_len:
                    if phrase not in candidates:
                        candidates[phrase] = {"keyword": phrase, "freq": 0, "pos": ent.start, "type": "entity"}
                    candidates[phrase]["freq"] += 1

        # Score and sort
        result = []
        for data in candidates.values():
            score = self._score(
                data["keyword"],
                data["freq"],
                total,
                [data["pos"]],
                is_bigram=" " in data["keyword"],
            )
            result.append({
                "keyword": data["keyword"],
                "score": round(score, 4),
                "freq": data["freq"],
                "type": data["type"],
            })

        result.sort(key=lambda x: x["score"], reverse=True)
        return result[: self.top_n]


# ------------------------------------------------------------------ #
# Demo
# ------------------------------------------------------------------ #

if __name__ == "__main__":
    extractor = KeywordExtractor(top_n=10, include_bigrams=True)

    examples = [
        "Machine learning is transforming the healthcare industry.",
        (
            "Deep learning models are being used in natural language processing "
            "and computer vision applications. Neural networks have revolutionized "
            "image recognition and sentiment analysis tasks."
        ),
        (
            "Python is the most popular programming language for data science. "
            "Libraries like Pandas, NumPy, and Scikit-learn make machine learning "
            "accessible to every developer."
        ),
    ]

    for text in examples:
        print(f"\nInput:    {text[:80]}{'...' if len(text) > 80 else ''}")
        keywords = extractor.extract(text)
        print(f"Keywords: {', '.join(keywords)}")

        detailed = extractor.extract_detailed(text)
        print("Detailed:")
        for item in detailed:
            print(f"  {item['keyword']:<25}   freq={item['freq']}  type={item['type']}")


Input:    Machine learning is transforming the healthcare industry.
Keywords: machine learning, learning transforming, healthcare industry
Detailed:
  machine learning            freq=1  type=bigram
  learning transforming       freq=1  type=bigram
  healthcare industry         freq=1  type=bigram

Input:    Deep learning models are being used in natural language processing and computer ...
Keywords: deep learning, learning models, natural language, language processing, computer vision, vision applications, applications neural, neural networks, revolutionized image, image recognition
Detailed:
  deep learning               freq=1  type=bigram
  learning models             freq=1  type=bigram
  natural language            freq=1  type=bigram
  language processing         freq=1  type=bigram
  computer vision             freq=1  type=bigram
  vision applications         freq=1  type=bigram
  applications neural         freq=1  type=bigram
  neural networks             freq=1  type=bigra